#### Data Preparation

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import sys
!{sys.executable} -m pip install xgboost
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import shap
# Load data
df = pd.read_csv('../data/raw/MachineLearningRating_v3.txt', sep='|')

# Create target features
df['HadClaim'] = (df['TotalClaims'] > 0).astype(int)
df['ClaimSeverity'] = np.where(df['HadClaim'] == 1, df['TotalClaims'], np.nan)

# Drop useless columns and high-cardinality IDs
df.drop(columns=['PolicyID', 'UnderwrittenCoverID'], inplace=True)

# Handle missing values
df = df.dropna(thresh=len(df)*0.5, axis=1)  # Drop columns with >50% missing
df = df.dropna(subset=['TotalClaims', 'TotalPremium', 'CalculatedPremiumPerTerm'])

# Fill remaining NaNs
df.fillna(df.median(numeric_only=True), inplace=True)


#### Feature Engineering & Encoding

In [ ]:
# Select features
features = ['Gender', 'Province', 'PostalCode', 'VehicleType', 'Make', 'Model',
            'Cubiccapacity', 'Kilowatts', 'RegistrationYear', 'NewVehicle',
            'CustomValueEstimate', 'AlarmImmobiliser', 'TrackingDevice']

# Drop high-cardinality or dirty columns
df = df[features + ['TotalClaims', 'CalculatedPremiumPerTerm', 'HadClaim', 'ClaimSeverity']]

# One-hot encode categorical features
df = pd.get_dummies(df, drop_first=True)

# Train-test split
X = df[df['HadClaim'] == 1].drop(columns=['TotalClaims', 'CalculatedPremiumPerTerm', 'HadClaim', 'ClaimSeverity'])
y = df[df['HadClaim'] == 1]['TotalClaims']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


#### Model Building – Regression (Claim Severity)

In [ ]:
linreg = LinearRegression()
linreg.fit(X_train, y_train)
y_pred_lr = linreg.predict(X_test)

print("Linear Regression R²:", r2_score(y_test, y_pred_lr))
print("Linear Regression RMSE:", mean_squared_error(y_test, y_pred_lr, squared=False))


In [ ]:
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest R²:", r2_score(y_test, y_pred_rf))
print("Random Forest RMSE:", mean_squared_error(y_test, y_pred_rf, squared=False))


In [ ]:
xgb = XGBRegressor(random_state=42)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

print("XGBoost R²:", r2_score(y_test, y_pred_xgb))
print("XGBoost RMSE:", mean_squared_error(y_test, y_pred_xgb, squared=False))


#### Interpretability with SHAP (Top 10 Features)

In [ ]:
explainer = shap.Explainer(xgb)
shap_values = explainer(X_test)

shap.summary_plot(shap_values, X_test, max_display=10)


#### Claim Probability Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Prepare data
X_bin = df.drop(columns=['TotalClaims', 'CalculatedPremiumPerTerm', 'ClaimSeverity'])
y_bin = df['HadClaim']

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_bin, y_bin, test_size=0.3, random_state=42)

clf = RandomForestClassifier()
clf.fit(X_train_b, y_train_b)
y_pred_b = clf.predict(X_test_b)

print("Accuracy:", accuracy_score(y_test_b, y_pred_b))
print("Precision:", precision_score(y_test_b, y_pred_b))
print("Recall:", recall_score(y_test_b, y_pred_b))
print("F1 Score:", f1_score(y_test_b, y_pred_b))
